# 알약 각인 인식 실험 — 코랩 실행용

**시작 전:** 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후 저장하세요.

위에서부터 셀을 하나씩 실행(▶ 또는 Shift+Enter)하면 됩니다.

## 1. GPU 확인
`True`가 나오면 GPU를 쓸 수 있어요. `False`면 위의 런타임 설정을 다시 확인하세요.

In [ ]:
import torch
print('GPU 사용 가능:', torch.cuda.is_available())

## 2. 실험 파일 올리기
셀을 실행하면 '파일 선택' 버튼이 나와요. **`dl_deep_imgPs_업로드용.zip`** 을 골라주세요.

(GitHub에 올라가 있으면 이 셀 대신 `!git clone https://github.com/yewon51019/dl_deep_imgPs` 를 써도 돼요.)

In [ ]:
from google.colab import files
up = files.upload()
zip_name = list(up)[0]
!unzip -o -q "$zip_name" -d project
%cd project/imprint_ocr
!ls

## 3. 설치 (EasyOCR + 한글 폰트)
1~2분 걸려요. 글자 인식 모델(약 100MB)은 4번 실행할 때 자동으로 받아져요.

In [ ]:
!pip install -q easyocr
!apt-get -qq install -y fonts-nanum > /dev/null
!python pill_imprint.py check

## 4. 실험 실행
`--gpu` 를 붙여서 GPU로 돌려요. 전처리 5가지를 모두 비교해요.

In [ ]:
!python pill_imprint.py run --gpu --methods A B C D AUTO --out results_easyocr

## 5. 결과 보기

In [ ]:
from IPython.display import Image, display
import pandas as pd
display(Image('results_easyocr/chart_summary.png'))
display(pd.read_csv('results_easyocr/summary.csv'))
display(pd.read_csv('results_easyocr/by_type.csv'))

## 6. 결과 내려받기
결과 폴더를 zip으로 묶어서 내 컴퓨터로 받아요.

In [ ]:
!zip -q -r results_easyocr.zip results_easyocr
files.download('results_easyocr.zip')

---
## (선택) 사광 사진 합성
위·오른쪽·아래·왼쪽 순서로 찍은 사진 4장을 올린 뒤 실행해요. 파일 이름 앞에 `1_`, `2_`, `3_`, `4_` 를 붙여두면 순서가 맞아요.

In [ ]:
import os
os.makedirs('shots', exist_ok=True)
up = files.upload()
for n, d in up.items():
    open(os.path.join('shots', n), 'wb').write(d)
!python pill_imprint.py fuse --inputs "shots/*" --out fused
!python pill_imprint.py crop --input fused/fused_min.png --out crops_fused
display(Image('fused/fused_min.png', width=400))
display(Image('crops_fused/crop_overview.png', width=400))

잘라낸 사진(`crops_fused/crop_00.png` 등)을 `labels.csv` 에 한 줄씩 추가하고 4번을 다시 실행하면 돼요. 왼쪽 폴더 아이콘에서 `labels.csv` 를 더블클릭하면 바로 편집할 수 있어요.